# Does the head-pairing entropy excursion appear in *flat* variants?

`attention_head_pairing.ipynb` found, on three variants with a late loss-spike event, that the
`=`-query attention entropy splits **2+2**: the two **distinct (solo) frequency** heads carry a
co-moving excursion while the **doubled-frequency** pair stays sharp. That has two separable parts:

- **static** 2+2 QK frequency redundancy (one frequency doubled across two heads) — expected to be a
  *universal* property of any 4-head model;
- **dynamic** entropy excursion — possibly specific to variants that destabilize.

This notebook tests both on six **flat** variants (no loss spike, from `escape_regime_summary.ipynb`):
p103/s485/ds598, p103/s999/ds598, p109/s485/ds999, p59/s999/ds598, p109/s485/ds42, p109/s999/ds42.

*Reference (the event variants, from the pairing notebook): max per-head entropy excursion ≈
0.32–0.40, within-pair correlation 0.85–0.98.*

In [1]:
import os
from itertools import combinations
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "modulo_addition_1layer").exists())
os.chdir(root)
from miscope.families.discovery import load_family_from_dir

fam = load_family_from_dir("data/modulo_addition_1layer", "data")
SPECS = [("p103/s485/ds598", dict(prime=103, seed=485, data_seed=598)),
         ("p103/s999/ds598", dict(prime=103, seed=999, data_seed=598)),
         ("p109/s485/ds999", dict(prime=109, seed=485, data_seed=999)),
         ("p59/s999/ds598",  dict(prime=59,  seed=999, data_seed=598)),
         ("p109/s485/ds42",  dict(prime=109, seed=485, data_seed=42)),
         ("p109/s999/ds42",  dict(prime=109, seed=999, data_seed=42))]
variants = {tag: fam.get_variant(**kw) for tag, kw in SPECS}
EQ_QUERY = 2
POSTGROK = 10000
PLATEAU = 20000
UNIFORM = {tag: float(np.log(3)) for tag in variants}   # uniform over (a, b, =)
variants

/home/megano/projects/mechinterp/miscope/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'p103/s485/ds598': Variant(family='modulo_addition_1layer', name='p103_seed485_dseed598', state=analyzed),
 'p103/s999/ds598': Variant(family='modulo_addition_1layer', name='p103_seed999_dseed598', state=analyzed),
 'p109/s485/ds999': Variant(family='modulo_addition_1layer', name='p109_seed485_dseed999', state=analyzed),
 'p59/s999/ds598': Variant(family='modulo_addition_1layer', name='p59_seed999_dseed598', state=analyzed),
 'p109/s485/ds42': Variant(family='modulo_addition_1layer', name='p109_seed485_dseed42', state=analyzed),
 'p109/s999/ds42': Variant(family='modulo_addition_1layer', name='p109_seed999_dseed42', state=analyzed)}

## 1. Per-head entropy trajectories (the six flat variants)

Same lens as the pairing notebook: per-head Shannon entropy of attention at the `=` query, mean over
the grid. Look for the 2+2 paired signature (a co-moving rise in two heads) vs genuine flatness.

In [2]:
def equals_query_entropy(variant, epochs, query=EQ_QUERY):
    p = variant.params["prime"]
    probe = variant.make_probe([[a, b] for a in range(p) for b in range(p)])
    rows = []
    for e in epochs:
        _, cache = variant.run_with_cache(probe, epoch=int(e), dtype=torch.float64)
        pat = cache["blocks.0.attn.hook_pattern"][:, :, query, :].detach().cpu().numpy()  # (B, H, K)
        rows.append((-(pat * np.log(np.clip(pat, 1e-12, None))).sum(-1)).mean(0))
    return np.array(rows)


traj = {}
for tag, v in variants.items():
    eps = np.array(v.get_available_checkpoints())
    traj[tag] = (eps, equals_query_entropy(v, eps))
    print(f"{tag}: {len(eps)} checkpoints")

p103/s485/ds598: 403 checkpoints


p103/s999/ds598: 352 checkpoints


p109/s485/ds999: 403 checkpoints


p59/s999/ds598: 352 checkpoints


p109/s485/ds42: 352 checkpoints


p109/s999/ds42: 352 checkpoints


In [3]:
for tag, (eps, ent) in traj.items():
    fig = go.Figure()
    fig.add_hline(y=float(np.log(3)), line=dict(dash="dot", color="gray"),
                  annotation_text="ln 3 (uniform)", annotation_position="top right")
    for h in range(ent.shape[1]):
        fig.add_trace(go.Scatter(x=eps, y=ent[:, h], mode="lines", name=f"head {h}"))
    fig.add_trace(go.Scatter(x=eps, y=ent.mean(1), mode="lines", name="mean",
                             line=dict(color="black", width=3)))
    fig.update_layout(title=f"{tag} (flat) — attention entropy at the `=` query",
                      xaxis_title="epoch", yaxis_title="entropy (nats)", height=360,
                      legend=dict(orientation="h", y=-0.25))
    fig.show()

## 2. Static pairing vs dynamic excursion — is the motif present?

For each flat variant: the **static** doubled-frequency QK redundancy (should be ~1 everywhere) and
the **dynamic** entropy excursion (max per-head magnitude, the co-moving pair and its within-pair
correlation, and whether that pair is the *solo*-frequency heads — the event-variant signature).

In [4]:
def head_qk_dominant_freq(variant, epoch):
    d = variant.artifacts.load_epoch("weight_basis_projection", int(epoch))
    P, f = d["attn_qk_fractional_power"], d["attn_qk_frequencies"]
    return f[(P.sum(2) + P.sum(1)).argmax(1)]


def qk_freq_cos(variant, epoch):
    P = variant.artifacts.load_epoch("weight_basis_projection", int(epoch))["attn_qk_fractional_power"]
    H = P.shape[0]; fp = P.reshape(H, -1)
    n = np.linalg.norm(fp, axis=1, keepdims=True)
    return fp @ fp.T / (n @ n.T)


def plateau_epoch(variant, target=PLATEAU):
    eps = np.array(variant.get_available_checkpoints())
    return int(eps[np.argmin(np.abs(eps - target))])


print(f"{'model':16} {'head freq':22} | {'doubled pair':12} {'QKcos':>6} | "
      f"{'exc pair':9} {'within':>7} {'max_exc':>8} {'pair=solo?':>10}")
for tag, (eps, ent) in traj.items():
    v = variants[tag]; plat = plateau_epoch(v)
    hf = head_qk_dominant_freq(v, plat); H = len(hf)
    S = qk_freq_cos(v, plat)
    same = [(i, j) for i, j in combinations(range(H), 2) if hf[i] == hf[j]]
    doubled = max(same, key=lambda ij: S[ij]) if same else None
    solo = {h for h in range(H) if list(hf).count(hf[h]) == 1}
    m = eps >= POSTGROK
    E = ent[m]; base = np.median(E, 0); exc = np.abs(E - base).max(0)
    Ccorr = np.corrcoef(E.T)
    exc_pair = tuple(sorted(int(h) for h in np.argsort(exc)[-2:]))
    within = Ccorr[exc_pair[0], exc_pair[1]]
    is_solo = set(exc_pair) <= solo
    dbl_str = f"{doubled} f{hf[doubled[0]]}" if doubled else "none"
    qkc = S[doubled] if doubled else float("nan")
    print(f"{tag:16} {str(hf.tolist()):22} | {dbl_str:12} {qkc:6.3f} | "
          f"{str(exc_pair):9} {within:7.2f} {exc.max():8.3f} {str(is_solo):>10}")

model            head freq              | doubled pair  QKcos | exc pair   within  max_exc pair=solo?
p103/s485/ds598  [40, 40, 24, 6]        | (0, 1) f40    0.993 | (2, 3)       0.99    0.167       True


p103/s999/ds598  [25, 6, 10, 6]         | (1, 3) f6     0.978 | (0, 2)       0.96    0.034       True


p109/s485/ds999  [30, 30, 20, 7]        | (0, 1) f30    0.998 | (2, 3)       0.67    0.181       True
p59/s999/ds598   [5, 21, 21, 15]        | (1, 2) f21    0.996 | (0, 3)       0.12    0.040       True
p109/s485/ds42   [7, 36, 7, 36]         | (0, 2) f7     1.000 | (2, 3)       0.73    0.034      False


p109/s999/ds42   [29, 29, 54, 5]        | (0, 1) f29    0.996 | (2, 3)      -0.51    0.142       True


## Reading

**The static pairing is universal.** Every flat variant carries a doubled-frequency QK pair with cosine ≈ 0.98–1.0 — identical to the event variants. The 2+2 frequency redundancy (one frequency shared by two near-identical QK heads) is a **general property of 4-head modular-addition models**, present whether or not the model ever destabilizes. (One variant, **p109/s485/ds42** — `[7,36,7,36]`, i.e. *two* doubled frequencies and **no solo head at all** — shows essentially no excursion, consistent with the picture below.)

**The dynamic excursion is a spectrum, not a binary.** Across the "flat" set the max per-head entropy excursion runs from event-like down to genuinely flat — and *where it appears, it lands on the solo (distinct-frequency) heads*, the same signature as the loss-spike variants:

- **p103/s485/ds598** (max-exc 0.167, within 0.99) and **p109/s485/ds999** (0.181, within 0.67) show a clear *paired* solo-head excursion with **no loss spike** — a slow version of the event motif.
- **p103/s999/ds598** shows a faint but cleanly paired drift (0.034, within 0.96).
- **p59/s999/ds598** is genuinely flat (0.040, within 0.12 — no pairing).
- **p109/s999/ds42** excurses on its two solo heads but **anti-phase** (0.142, within −0.51) — the solo heads move, but oppositely rather than together: a degraded version of the co-moving motif.

So "escape regime" vs "flat" is better read as a **continuum of the same solo-head entropy dynamics**, with the loss spike at the sharp end — not two categories. The redundant (doubled) pair holds steady throughout; what varies between runs is how hard, and how *coherently*, the solo-frequency heads de-sharpen (within-pair correlation ranging 0.99 → 0.12 → −0.51).

**Caveats.** Six runs; "flat" is a loss-spike classification, and the entropy excursion is read off the same float64 forward as elsewhere. Eyeball "flatness" and the quantitative max-excursion can disagree (e.g. p109/s485/ds999 reads flat but carries a moderate paired excursion) — the curves and the table together are the honest read. Each prime+seed+data_seed is a unique model run.